# 11 · Beyond puzzles: held-out game positions and full games

Puzzles measure one skill: finding the single winning move in a tactical position. Our progress report committed to two more evaluations, which this notebook runs for **every model** (the base model, the three first models, and every new run):

**1. Held-out game positions.** 1,000 ordinary positions from games between 2000+ players that no model trained on (the validation games, kept separate by game). In most such positions several moves are reasonable, so "solved or not" does not apply. Instead we measure:

| Metric | Meaning |
|---|---|
| Legal (%) | answers that are legal moves |
| ACPL (cp) | average centipawn loss of the legal answers vs. Stockfish's best move (lower is better) |
| Blunders (%) | answers that are legal but lose 200+ centipawns (two pawns), the threshold DPO uses |
| Same as Stockfish (%) | answers equal to Stockfish's best move |
| Same as the player (%) | answers equal to the move the 2000+ player actually made |

This separates two ways a model can improve, which puzzles cannot tell apart: **making fewer blunders** in ordinary positions vs. **finding tactics** in puzzles. A random legal move is included as a reference row.

**2. Full games.** Each model plays 50 games against each of two opponents:
- **a random mover**, which plays a uniformly random legal move. This is the weakest possible opponent: a model that plays sensibly should beat it.
- **Stockfish at depth 1**, a very shallow search that is still far stronger than our models. Most models will lose almost every game; the interesting differences may be in draws and in how long they survive.

Every game starts after the first three moves of a held-out game (25 different openings), and each opening is played once with each color. The model gets the usual prompt and greedy decoding. If its answer is illegal or unparseable, a random legal move is played for it and the mistake is counted. This lets every game finish, so the game result measures move quality and the **invalid-move rate** measures legality over a whole game. A game still going after 100 moves counts as a draw.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run.
3. Set `QUICK_CHECK = False` and run the real thing. For a run this long, use a **Job** so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification.
4. Afterwards, commit `results/` from the Git folder; Databricks cannot commit notebook outputs. Then **File → Export → IPython Notebook**, replace this notebook with the exported file on a laptop, and commit it there, so the figures also show inside the notebook on GitHub.

**Time:** about 10 minutes per model, so 1–1.5 hours for all nine models. Models that already have results are skipped, so the notebook can be rerun after new models are trained.

**Outputs**
- the frozen evaluation sets `data/eval/game_positions.csv` and `data/eval/game_openings.csv` (on GitHub, so every model is evaluated on the same positions)
- per model: `results/<model>_positions.json` and `results/<model>_games.json` (every answer and every game, with its moves)
- summaries: `results/positions_summary.csv` and `results/games_summary.csv`
- figures and tables in `results/figures/game_evals/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks(require_gpu=True)  # stops with a clear message if no GPU is attached

In [ ]:
# Settings
QUICK_CHECK = True    # True = 10 positions and 2 openings for two models, to check that everything works. Saves nothing to results/.
REDO = False          # True = evaluate models again even if they already have results
MODELS = {            # model name -> its adapter folder (None = the base model). Missing folders are skipped.
    "base": None,
    "sft_v1": "checkpoints/sft_v1",
    "dpo_v1": "checkpoints/dpo_v1",
    "grpo_v1": "checkpoints/grpo_v1",
    "sft_stockfish": "checkpoints/sft_stockfish",
    "dpo_v2": "checkpoints/dpo_v2",
    "dpo_v3": "checkpoints/dpo_v3",
    "grpo_v2": "checkpoints/grpo_v2",
    "grpo_v3": "checkpoints/grpo_v3",
}

import gc
import json

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import (figures_dir, grade_game_positions, play_against_opponents, random_move_positions,
                                 save_table, update_summary)
from chesslm.play import opening_lines

sft_cfg, prompt_cfg, eval_cfg = load_config("sft"), load_config("prompt"), load_config("eval")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
pos_cfg, games_cfg = eval_cfg["game_positions"], eval_cfg["games"]
FIGURES_NAME = "game_evals"

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    MODELS = {name: MODELS[name] for name in ["base", "sft_v1"]}
    games_cfg["max_moves"] = 15
    FIGURES_NAME += "_check"
FIGURES = figures_dir(FIGURES_NAME)

## 1. The frozen evaluation sets

Built once from the validation games (which no model trained on), with a fixed seed, and saved in `data/eval/`. After that they are only loaded, so every model, now and later, is evaluated on exactly the same positions and openings.

In [ ]:
positions_file, openings_file = REPO_ROOT / pos_cfg["file"], REPO_ROOT / games_cfg["openings_file"]
if not positions_file.exists() or not openings_file.exists():
    val = pd.read_parquet(REPO_ROOT / "data" / "processed" / "val.parquet")
    val.sample(n=pos_cfg["n_positions"], random_state=pos_cfg["seed"]).to_csv(positions_file, index=False)
    lines = opening_lines(val, games_cfg["n_openings"], games_cfg["opening_plies"], games_cfg["seed"])
    pd.DataFrame({"opening_uci": lines}).to_csv(openings_file, index=False)
    print("Built and saved the evaluation sets.")

positions = pd.read_csv(positions_file)
openings = list(pd.read_csv(openings_file).opening_uci)
if QUICK_CHECK:
    positions, openings = positions.head(10), openings[:2]
print(f"{len(positions):,} held-out positions from {positions.game_id.nunique():,} games | "
      f"{len(openings)} openings x 2 colors = {2 * len(openings)} games per opponent")

## 2. Evaluate every model

For each model: grade its answers on the held-out positions, then play the games. Each model prints one line per evaluation.

In [ ]:
def has_results(name):
    return all((experiments.RESULTS / f"{name}_{kind}.json").exists() for kind in ["positions", "games"])

for name, adapter in MODELS.items():
    if adapter and not (REPO_ROOT / adapter).exists():
        print(f"{name}: skipped, {adapter} not found (train it first)")
        continue
    if has_results(name) and not REDO:
        print(f"{name}: already evaluated (set REDO = True to evaluate again)")
        continue
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=REPO_ROOT / adapter if adapter else None)
    grade_game_positions(model, tokenizer, name, configs, positions)
    play_against_opponents(model, tokenizer, name, configs, openings)
    del model
    gc.collect()
    torch.cuda.empty_cache()

The reference row: the same position metrics for a player that picks a random legal move. It needs only Stockfish, no GPU.

In [ ]:
REFERENCE = "random legal move"
summary_path = experiments.RESULTS / "positions_summary.csv"
if REDO or not summary_path.exists() or REFERENCE not in set(pd.read_csv(summary_path).run):
    update_summary("positions_summary", ["run"],
                   {"run": REFERENCE, **random_move_positions(positions, eval_cfg, workers=4, seed=pos_cfg["seed"])})

## 3. Results on held-out game positions

**How to read this:** lower ACPL and fewer blunders mean better moves in ordinary positions. "Same as Stockfish" is the closest thing to accuracy here. "Same as the player" shows how human-like the moves are: `sft_v1` learned from human moves and `sft_stockfish` from Stockfish's, so these two columns should move in opposite directions between them. Compare every model with the random-move row.

In [ ]:
POSITION_COLUMNS = {"legal_rate": "Legal (%)", "acpl": "ACPL (cp)", "blunder_rate": "Blunders (%)",
                    "stockfish_match_rate": "Same as Stockfish (%)", "player_match_rate": "Same as the player (%)",
                    "illegal_rate": "Illegal (%)", "unparseable_rate": "Unparseable (%)"}
order = [name for name in list(MODELS) + [REFERENCE]]
pos_summary = pd.read_csv(experiments.RESULTS / "positions_summary.csv").set_index("run")
pos_table = pos_summary.loc[[n for n in order if n in pos_summary.index], list(POSITION_COLUMNS)]
pos_table = pos_table.rename(columns=POSITION_COLUMNS)
percent = [c for c in pos_table.columns if c != "ACPL (cp)"]
pos_table[percent] = (pos_table[percent] * 100).round(1)
pos_table["ACPL (cp)"] = pos_table["ACPL (cp)"].round(0)
pos_table.index.name = "Model"
save_table(pos_table, FIGURES, "game_positions",
           f"{len(positions):,} held-out positions from games between 2000+ players. Blunder = a legal answer losing "
           f"{pos_cfg['blunder_cp']}+ centipawns; its rate is over all answers. ACPL is over legal answers only.")
display(pos_table)

plots.metric_panels(pos_table, [("Legal (%)", "legal answers (%)", "{:.0f}"),
                                ("ACPL (cp)", "ACPL (cp, lower is better)", "{:.0f}"),
                                ("Blunders (%)", "blunders (% of answers)", "{:.0f}"),
                                ("Same as Stockfish (%)", "same as Stockfish (%)", "{:.0f}")],
                    FIGURES / "game_positions.png", title="Held-out game positions: legality and move quality");

## 4. Results in full games

**How to read this:** the **score** is the usual chess score, (wins + ½ × draws) / games; 50% means even. The **invalid-move rate** is the share of the model's moves that were illegal or unparseable (and replaced by a random move). Against the random mover, a model that plays reasonable chess should score well above 50%. Against Stockfish, any draw or win is notable.

In [ ]:
games_summary = pd.read_csv(experiments.RESULTS / "games_summary.csv")
GAME_COLUMNS = {"wins": "Wins", "draws": "Draws", "losses": "Losses", "score": "Score (%)",
                "invalid_move_rate": "Invalid moves (%)", "average_game_length": "Average length (plies)"}
for opponent in games_cfg["opponents"]:
    rows = games_summary[games_summary.opponent == opponent].set_index("run")
    table = rows.loc[[n for n in MODELS if n in rows.index]]
    readable = table[list(GAME_COLUMNS)].rename(columns=GAME_COLUMNS)
    readable["Score (%)"] = (readable["Score (%)"] * 100).round(1)
    readable["Invalid moves (%)"] = (readable["Invalid moves (%)"] * 100).round(1)
    readable["Average length (plies)"] = readable["Average length (plies)"].round(0)
    readable.index.name = "Model"
    opponent_name = "a random mover" if opponent == "random" else f"Stockfish (depth {games_cfg['stockfish_depth']})"
    save_table(readable, FIGURES, f"games_vs_{opponent}",
               f"{int(table.games.iloc[0])} games per model against {opponent_name}, each opening once with each color. "
               "Invalid moves were replaced by a random legal move. Score = (wins + draws/2) / games.")
    print(f"Against {opponent_name}:")
    display(readable)
    plots.game_results(table, FIGURES / f"games_vs_{opponent}.png", title=f"Game results against {opponent_name}");

To look at individual games, each game's moves are saved in standard PGN format in `results/<model>_games.json`, which any chess website (e.g. the Lichess analysis board) can replay. The first game of the first model:

In [ ]:
first = next(name for name in MODELS if (experiments.RESULTS / f"{name}_games.json").exists())
with open(experiments.RESULTS / f"{first}_games.json") as f:
    game = json.load(f)["games"][games_cfg["opponents"][0]][0]
print(f"{first}, as {game['model_color']}: {game['result']} ({game['end_reason']}), "
      f"{game['illegal'] + game['unparseable']} invalid moves\n")
print(game["pgn"])

## Summary of outputs

- `results/figures/game_evals/`: `game_positions` (table and figure), `games_vs_random` and `games_vs_stockfish` (tables and figures)
- `results/positions_summary.csv`, `results/games_summary.csv`: one row per model (and opponent)
- `results/<model>_positions.json`, `results/<model>_games.json`: every answer and every game
- To add a model trained later (e.g. a combination from notebook 12), add it to `MODELS` and rerun: models that already have results are skipped.